# Assignment MMB

---

## Important Dates
- Homework release: Tue 22 Sep 2026
- **Homework due:** Mon 30 Nov 2026, 11:59 AM

---

This notebook is not graded, the markdown cells should help you create an error component model, step by step.

## Some rules
1. You are allowed to use any built-in Python library that comes with Anaconda. If you want to use an external library, you may do so, but must justify your choice.

2. Make sure you use the `data` folder provided in the repository in read-only mode. (Or alternatively, be sure you don’t change any of the files.) One should be able to reproduce your results using the original data files.

3. Be sure to provide a concise textual description of your thought process, the assumptions you made, the solution you implemented, and explanations for your answers.

4. Several questions ask to comment and analyze the results. Make sure to provide a clear interpretation of the results. A few lines of text should be enough to explain your reasoning and the implications of the results.

5. Please write all your comments in **English**, and use meaningful variable names in your code. 

6. Make sure to print results and/or dataframes that confirm you have properly addressed the task.


In [2]:
import biogeme.version as ver
print(ver.get_text())
# This assignment uses Biogeme 3.3.5.

biogeme 3.3.5 [2026-10-10]
Home page: http://biogeme.epfl.ch
Submit questions to https://groups.google.com/d/forum/biogeme
Michel Bierlaire, Transport and Mobility Laboratory, Ecole Polytechnique Fédérale de Lausanne (EPFL)



In [37]:
import pandas as pd
import biogeme.database as db
import biogeme.biogeme as bio
from biogeme.models import logit, loglogit, lognested, logcnl
from biogeme.nests import OneNestForNestedLogit, NestsForNestedLogit, OneNestForCrossNestedLogit, NestsForCrossNestedLogit
from biogeme.expressions import Beta, Variable, log, exp, RandomVariable, IntegrateNormal, Draws, MonteCarlo, Derive
from biogeme.results_processing import compile_estimation_results, get_pandas_estimated_parameters
from biogeme.tools import likelihood_ratio_test
from datetime import datetime as dt
print(f'Date: {dt.now().strftime("%Y-%m-%d %H:%M:%S")}')

import numpy as np
from scipy.stats import chi2

Date: 2026-10-10 08:00:26


# Model 0 

---

Start with a model specification that includes alternative specific constants, and cost and travel time of the different alternatives associated with generic parameters. Report both the specification (i.e., the utility functions) and the estimation results (parameter values, t-tests or p-values, null and final log likelihoods).  

Calculate the total public transport duration as the sum of `dur_pt_access`, `dur_pt_rail`, `dur_pt_bus`, and `dur_pt_int`. Similarly, calculate the total driving cost as the sum of `cost_driving_fuel` and `cost_driving_ccharge`.

1. Comment on the estimation output (sign and significance of all parameters, and goodness of fit indices).  

---

**0.1. – Preparing the data**  
- Load your assigned LPMC dataset.
- Compute `total_pt_duration = dur_pt_access + dur_pt_rail + dur_pt_bus + dur_pt_int`.
- Compute `total_driving_cost = cost_driving_fuel + cost_driving_ccharge`.
- All individuals and all alternatives should be kept.

In [4]:
# Load your assigned data file. Replace the placeholder with its filename.
df = pd.read_table('lpmc25.dat')

# Create the two aggregate attributes requested in the question.
df['total_pt_duration'] = df['dur_pt_access'] + df['dur_pt_rail'] + df['dur_pt_bus'] + df['dur_pt_int']
df['total_driving_cost'] = df['cost_driving_fuel'] + df['cost_driving_ccharge']

# Create a Biogeme database from your dataframe.
database = db.Database('db25', df)

**0.2. – Model specification**  
- Define a model with:
  - Alternative-specific constants (ASCs),
  - A generic parameter for total cost,
  - A generic parameter for total travel time.
- Use the same parameter values across alternatives (generic parameters).


In [5]:
# Choice variable: 1=Walk, 2=Cycle, 3=PT, 4=Drive/Car
travel_mode = Variable('travel_mode')

# Mode travel durations (in minutes or hours as provided)
dur_walking = Variable('dur_walking')
dur_cycling = Variable('dur_cycling')
total_pt_duration = Variable('total_pt_duration')
dur_driving = Variable('dur_driving')

# Mode costs
cost_transit = Variable('cost_transit')
total_driving_cost = Variable('total_driving_cost')

In [6]:
# Alternative Specific Constants (ASCs)
# Walk (mode 1) is chosen as the reference alternative, so its ASC is normalized to 0 (status=1)
asc_walk  = Beta('asc_walk', 0, None, None, 1)   # Fixed for identification
asc_cycle = Beta('asc_cycle', 0, None, None, 0)  # Estimated
asc_pt    = Beta('asc_pt', 0, None, None, 0)     # Estimated
asc_car   = Beta('asc_car', 0, None, None, 0)    # Estimated

# Generic taste coefficients across all modes for Model 0
b_time = Beta('b_time', 0, None, None, 0)
b_cost = Beta('b_cost', 0, None, None, 0)

In [7]:
# Write one utility for each alternative, then collect them in V_0.

# 1: Walk (reference mode: asc_walk is fixed to 0, no monetary cost)
V_walk_0 = asc_walk + b_time * dur_walking

# 2: Cycle (no monetary cost)
V_cycle_0 = asc_cycle + b_time * dur_cycling

# 3: Public Transport (PT)
V_pt_0 = asc_pt + b_time * total_pt_duration + b_cost * cost_transit

# 4: Car / Driving
V_car_0 = asc_car + b_time * dur_driving + b_cost * total_driving_cost
V_0 = {
    1: V_walk_0,
    2: V_cycle_0,
    3: V_pt_0,
    4: V_car_0
}

**0.3. – Estimation of the parameters**  
- Estimate the model using Biogeme.
- Report:
  - Parameter estimates,
  - t-statistics or p-values,
  - Null and final log-likelihoods.

In [8]:
# Build the log-likelihood with loglogit, create a BIOGEME object, and estimate Model 0.

# All alternatives are available to all individuals in this assignment setup
av_0 = {
    1: 1,
    2: 1,
    3: 1,
    4: 1
}

# Define log-likelihood function
# - util: dictionary mapping alternative IDs to utility functions (V_0)
# - av: dictionary mapping alternative IDs to availability conditions (av_0)
# - i: observed choice variable (travel_mode)
logprob_0 = loglogit(util=V_0, av=av_0, i=travel_mode)

# Create Biogeme object
biogeme_0 = bio.BIOGEME(database, logprob_0)
biogeme_0.model_name = "model_0"

# Estimate the model
results_0 = biogeme_0.estimate()

In [9]:
# In Biogeme 3.3.5, the parameter tables are named: 
param_0 = get_pandas_estimated_parameters(estimation_results=results_0)
display(param_0['Estimated parameters'])

# Keep every estimated model in this dictionary for later comparisons.
all_results = {}
all_results['model_0'] = results_0

,Name,Value,Robust std err.,Robust t-stat.,Robust p-value
0,b_time,-5.076625,0.202460,-25.074754,0.000000e+00
1,asc_cycle,-4.408243,0.133120,-33.114707,0.000000e+00
2,asc_pt,-0.334560,0.054430,-6.146551,7.918606e-10
3,b_cost,-0.215469,0.014298,-15.070394,0.000000e+00
4,asc_car,-1.073406,0.079570,-13.490104,0.000000e+00


In [10]:
# In order to print general statistics such as the number of estimated parameters, the final log likelihood or the final gradient norm:

general_stats = results_0.get_general_statistics()
display(general_stats)

# or more synthetic
print(results_0.short_summary())

{'Number of estimated parameters': '5',
 'Sample size': '5000',
 'Excluded observations': '0',
 'Init log likelihood': '-4447.957',
 'Final log likelihood': '-4447.957',
 'Likelihood ratio test for the init. model': '-0',
 'Rho-square for the init. model': '0',
 'Rho-square-bar for the init. model': '-0.00112',
 'Akaike Information Criterion': '8905.914',
 'Bayesian Information Criterion': '8938.5',
 'Final gradient norm': '1.0627E-03',
 'Bootstrapping time': 'None'}

Results for model model_0
Nbr of parameters:		5
Sample size:			5000
Excluded data:			0
Final log likelihood:		-4447.957
Akaike Information Criterion:	8905.914
Bayesian Information Criterion:	8938.5



**0.4. – Interpretation of the results**  
- Discuss the sign of each parameter and whether it aligns with expected behavior.
- Comment on the statistical significance of the coefficients.
- Evaluate the overall model fit.

### Interpretation of Model 0 Results

#### 1. Estimation Summary and Signs
* **Travel Time (`b_time = -5.08`):** The coefficient is strictly negative, which is behaviorally consistent with microeconomic theory: an increase in travel duration imposes disutility on travelers.
* **Travel Cost (`b_cost = -0.215`):** The coefficient is strictly negative, reflecting the expected disutility of out-of-pocket monetary expenditures.
* **Statistical Significance:** Both attributes display very large absolute $t$-statistics ($\vert{}t\vert{} = 25.07$ for time and $\vert{}t\vert{} = 15.07$ for cost, with $p < 0.001$), indicating that both travel time and travel cost are highly significant predictors of mode choice.

#### 2. Alternative-Specific Constants (ASCs)
* Walk is normalized as the reference alternative ($ASC_{walk} = 0$).
* **Cycle (`asc_cycle = -4.41`):** Shows a substantial negative constant ($t = -33.11$), reflecting an intrinsic negative baseline preference for cycling compared to walking when time and cost are held equal. This aligns with low observed bicycle mode share in London.
* **Public Transport (`asc_pt = -0.33`):** Exhibits a slight negative baseline constant ($t = -6.15$), capturing unobserved disutilities associated with transit (e.g., waiting, crowding, transfers) relative to walking.
* **Car (`asc_car = -1.07`):** Displays a negative constant ($t = -13.49$), indicating that, all else held equal, travelers have a lower baseline preference for driving over walking for equivalent trips.

#### 3. Behavioral Implication: Value of Time (VOT)
Under this generic specification, the implied trade-off between travel time and travel cost (the Value of Travel Time Savings) is constant across all modes:
$$\text{VOT} = \frac{\frac{\partial U}{\partial \text{time}}}{\frac{\partial U}{\partial \text{cost}}} = \frac{\beta_{\text{time}}}{\beta_{\text{cost}}} = \frac{-5.077}{-0.215} \approx 23.56 \text{ £/hour}$$
This represents an average willingness to pay of approximately **£23.56 per hour saved**, which is a plausible figure for the greater London metropolitan area.

#### 4. Model Limitations and Justification for Model 1
Model 0 assumes a homogeneous marginal disutility of time ($\beta_{\text{time}}$) across all four modes. Behaviorally, this is restrictive because time spent cycling or walking involves physical exertion, driving involves active traffic stress, and public transit allows passive activities. This limitation motivates Model 1, where mode-specific time coefficients will be introduced.

<div class="alert alert-info">
  Your comments should be concise and to the point, focusing on the most relevant aspects of the model and its results.
</div>

# Model 1 

---

Using Model 0 as the base model, include alternative-specific parameters (i.e., different parameters per alternative) for one of the attributes of Model 0. Report both the specification and the estimation results (parameter values, t-tests or p-values, null and final log likelihoods).

1. State the behavioral assumption of defining alternative-specific parameters in this specific situation.  
2. Comment on the estimation output (as defined previously, including any changes from the previous model).  

3. Compare Model 0 and Model 1 with an appropriate statistical test. Justify your choice of test. State the null hypothesis and the result of the test. Denote the preferred model as $\text{Model}_\text{pref}$.  

---

**1.1. – Model modification**  
- Start from Model 0 and choose **one attribute** (e.g., cost or time).
- Replace the generic coefficient for that attribute with **alternative-specific parameters**.
  - For example: instead of one $\beta_{\text{time}}$, define $\beta_{\text{time, car}}$...


In [11]:
# Choose ONE attribute from Model 0 (for example, time or cost).
# Replace its generic coefficient with alternative-specific coefficients.
# Rebuild the utilities affected by your choice and collect them in V_1.
# Keep Model 1 nested in Model 0 so that a likelihood-ratio test is valid.

''' 
V_i = ASC_i + (b_time + delta_time_i) * T_i + b_cost * C_i
where: Walking and cycling don't have monetary cost; delta_time_car = 0 (car is the reference mode)
'''

# Baseline time coefficient (same role as Model 0's b_time)
b_time = Beta('b_time', -5.08, None, None, 0)

# Mode-specific deviations from the Model0 time coefficient; Car's deviation is reference and set 0 
delta_time_walk  = Beta('delta_time_walk', 0, None, None, 0)
delta_time_cycle = Beta('delta_time_cycle', 0, None, None, 0)
delta_time_pt    = Beta('delta_time_pt', 0, None, None, 0)


# Cost coefficient stays generic (same as Model 0)
b_cost = Beta('b_cost', -0.215, None, None, 0)

# Alternative-specific constants (same as Model 0); Walk is the reference and asc_walk is 0
asc_walk  = Beta('asc_walk', 0, None, None, 1)
asc_cycle = Beta('asc_cycle', -4.41, None, None, 0)
asc_pt    = Beta('asc_pt', -0.335, None, None, 0)
asc_car   = Beta('asc_car', -1.07, None, None, 0)

# Utilities for Model 1
V_walk_1  = asc_walk + (b_time + delta_time_walk) * dur_walking
V_cycle_1 = asc_cycle + (b_time + delta_time_cycle) * dur_cycling
V_pt_1    = asc_pt + (b_time + delta_time_pt) * total_pt_duration + b_cost * cost_transit
V_car_1   = asc_car + b_time * dur_driving + b_cost * total_driving_cost

V_1 = {
    1: V_walk_1,
    2: V_cycle_1,
    3: V_pt_1,
    4: V_car_1
}

**1.2. – Behavioral interpretation**  
- Clearly state the behavioral assumption behind using different coefficients across alternatives.
  - Why might the sensitivity to that attribute differ by alternative?


In [17]:
# In your written answer, explain why sensitivity to your selected attribute
# may differ between transport modes in the context of your own data.

'''
Model 1 replaces the single generic time coefficient b_time by mode-specific coefficients (b_time + delta_i), 
with car as reference mode (delta_car = 0).

Assumption: the marginal disutility of travel time differs across modes because the same minute is perceived differently.
  - Walking / cycling: physical effort (and safety for cycling)
    may raise the perceived cost of time -> expect delta < 0.
  - Public transport: productive use may lowers it, but waiting,
    transfers and crowding may raise it -> sign of delta_pt ambiguous.
  - Car (reference): baseline b_time captures traffic stress.

 If the nonzero deltas are jointly significant (LR test in 1.4), Model 1 is preferred over Model 0.
'''

'\nModel 1 replaces the single generic time coefficient b_time by mode-specific coefficients (b_time + delta_i), \nwith car as reference mode (delta_car = 0).\n\nAssumption: the marginal disutility of travel time differs across modes because the same minute is perceived differently.\n  - Walking / cycling: physical effort (and safety for cycling)\n    may raise the perceived cost of time -> expect delta < 0.\n  - Public transport: productive use may lowers it, but waiting,\n    transfers and crowding may raise it -> sign of delta_pt ambiguous.\n  - Car (reference): baseline b_time captures traffic stress.\n\n If the nonzero deltas are jointly significant (LR test in 1.4), Model 1 is preferred over Model 0.\n'

**1.3. – Estimation and output analysis**  
- Estimate the updated model.
- Report and interpret:
  - Parameter estimates and their signs,
  - Statistical significance (t-stats or p-values),
  - Null and final log-likelihoods,
  - Any noticeable changes compared to Model 0.


In [19]:
'''
The built-in function in Biogeme:

def loglogit(util, av, i):
    util: dict, {alternative_id: utility_expression}
    av:   dict, {alternative_id: availability_expression}
    i:    variable or expression, observed choice
    returns: log-likelihood expression
    
The same as:

exp_util_j = av[j] * exp(util[j])
denominator = sum over j of av[j] * exp(util[j])
'''

'\nThe built-in function in Biogeme:\n\ndef loglogit(util, av, i):\n    util: dict, {alternative_id: utility_expression}\n    av:   dict, {alternative_id: availability_expression}\n    i:    variable or expression, observed choice\n    returns: log-likelihood expression\n    \nThe same as:\n\nexp_util_j = av[j] * exp(util[j])\ndenominator = sum over j of av[j] * exp(util[j])\n'

In [18]:
# Estimate Model 1 and report its short summary and estimated-parameters table.
# Use a distinct name such as 'model_1' and store its EstimationResults object.

'''
We reuse the same log-likelihood structure as Model 0, but with the new utility functions V (mode-specific time coefficients).
All alternatives are available (Cn = all modes).
'''
# available:1
av_1 = {1: 1, 2: 1, 3: 1, 4: 1}

# Build the log-likelihood for Model 1
logprob_1 = loglogit(util=V_1, av=av_1, i=travel_mode)

# Create the Biogeme object and give the model a name
biogeme_1 = bio.BIOGEME(database, logprob_1)
biogeme_1.model_name = "model_1"

# Estimate the model
results_1 = biogeme_1.estimate()

# --- Report the results ---
# 1) Estimated parameters (value, robust std err, t-stat, p-value)
param_1 = get_pandas_estimated_parameters(estimation_results=results_1)
print("Estimated parameters of Model 1:")
display(param_1['Estimated parameters'])

# 2) General statistics (number of params, final log-likelihood, AIC, BIC)
print("General statistics of Model 1:")
display(results_1.get_general_statistics())

# 3) Short summary
print(results_1.short_summary())

Estimated parameters of Model 1:


,Name,Value,Robust std err.,Robust t-stat.,Robust p-value
0,b_time,-6.901847,0.447735,-15.415017,0.000000
1,delta_time_walk,-0.935801,0.501767,-1.865013,0.062180
2,asc_cycle,-4.886802,0.239140,-20.434937,0.000000
3,delta_time_cycle,0.198088,0.672535,0.294539,0.768346
4,asc_pt,-2.083427,0.138895,-15.000022,0.000000
5,delta_time_pt,3.430396,0.253519,13.531131,0.000000
6,b_cost,-0.189650,0.016740,-11.329434,0.000000
7,asc_car,-1.610934,0.135474,-11.891053,0.000000


General statistics of Model 1:


{'Number of estimated parameters': '8',
 'Sample size': '5000',
 'Excluded observations': '0',
 'Init log likelihood': '-4447.967',
 'Final log likelihood': '-4107.343',
 'Likelihood ratio test for the init. model': '681.2494',
 'Rho-square for the init. model': '0.0766',
 'Rho-square-bar for the init. model': '0.0748',
 'Akaike Information Criterion': '8230.685',
 'Bayesian Information Criterion': '8282.823',
 'Final gradient norm': '3.0925E-05',
 'Bootstrapping time': 'None'}

Results for model model_1
Nbr of parameters:		8
Sample size:			5000
Excluded data:			0
Final log likelihood:		-4107.343
Akaike Information Criterion:	8230.685
Bayesian Information Criterion:	8282.823



In [20]:
# Add Model 1 before producing a comparison table.
all_results['model_1'] = results_1

'''
comparison_table, _ = compile_estimation_results(all_results)
display(comparison_table)
'''

# Build the comparison table (this line was commented out by mistake)
try:
    comparison_table, _ = compile_estimation_results(all_results)
    display(comparison_table)
except Exception as e:
    print("Could not build the comparison table:", e)
    # Fallback: print summaries individually
    for name, res in all_results.items():
        print(f"\n===== {name} =====")
        print(res.short_summary())

,model_0,model_1
Number of estimated parameters,5,8
Sample size,5000,5000
Final log likelihood,-4447.957,-4107.343
Akaike Information Criterion,8905.914,8230.685
Bayesian Information Criterion,8938.5,8282.823
b_time (t-test),-5.08 (-25.1),-6.9 (-15.4)
asc_cycle (t-test),-4.41 (-33.1),-4.89 (-20.4)
asc_pt (t-test),-0.335 (-6.15),-2.08 (-15)
b_cost (t-test),-0.215 (-15.1),-0.19 (-11.3)
asc_car (t-test),-1.07 (-13.5),-1.61 (-11.9)


**1.4. – Model comparison and selection**  
- Perform a likelihood ratio test (or another appropriate test) to compare Model 0 and Model 1.
- Clearly state:
  - The null hypothesis,
  - The degrees of freedom used,
  - The value of the test statistic and p-value,
  - The conclusion: which model is preferred ($\text{Model}_\text{pref}$)?

In [ ]:
# Model 0 is the restricted model and Model 1 is the unrestricted model.
# Extract each model's final_loglikelihood and number_of_parameters.
# Use likelihood_ratio_test to test whether the extra Model 1 parameters
# jointly improve fit. State the null hypothesis, degrees of freedom,
# statistic, threshold (or p-value), and your preferred model.

'''
Model 0 (restricted):   delta_time_walk = delta_time_cycle = delta_time_pt = 0
Model 1 (unrestricted): at least one delta != 0

H0: all three deltas are jointly zero
    (a single generic time coefficient is sufficient)
H1: at least one delta != 0

Test statistic: LR = -2 * (LL_0 - LL_1)
Under H0, LR ~ chi-squared with df = K_1 - K_0.
'''

In [23]:
# Extract log-likelihoods and number of estimated parameters
logL_0 = all_results['model_0'].final_loglikelihood
nparam_0 = all_results['model_0'].number_of_parameters # or number_of_free_parameters


logL_1 = all_results['model_1'].final_loglikelihood
nparam_1 = all_results['model_1'].number_of_parameters

# Run likelihood ratio test: model_0 is restricted, model_1 is unrestricted
lr_result = likelihood_ratio_test(
    model1=(logL_0, nparam_0),
    model2=(logL_1, nparam_1),
    significance_level=0.05
)
display(lr_result)

# Display results
print(f"Null hypothesis: Model 0 is sufficient (equal time sensitivity across modes)")
print(f"Degrees of freedom: {nparam_1 - nparam_0}")
print(f"Log-likelihood Model 0: {logL_0:.2f}")
print(f"Log-likelihood Model 1: {logL_1:.2f}")
print(f"Likelihood Ratio Statistic: {lr_result.statistic:.2f}")
print(f"Chi-squared Threshold (95%): {lr_result.threshold:.2f}")
print(f"{lr_result.message}")

LRTuple(message='H0 can be rejected at level 5.0%', statistic=681.2282986846767, threshold=np.float64(7.814727903251179))

Null hypothesis: Model 0 is sufficient (equal time sensitivity across modes)
Degrees of freedom: 3
Log-likelihood Model 0: -4447.96
Log-likelihood Model 1: -4107.34
Likelihood Ratio Statistic: 681.23
Chi-squared Threshold (95%): 7.81
H0 can be rejected at level 5.0%


# Model 2 

---

Using $\text{Model}_\text{pref}$ as the base model, choose a socioeconomic characteristic and define two model specifications. In the first specification, interact the socioeconomic characteristic with an alternative-specific constant. In the second specification, interact it with an alternative attribute. Report both the specification and the estimation results (parameter values, t-tests or p-values, null and final log likelihoods).

1. State the behavioral assumptions in this specific situation and how this specification explains behavior differently from $\text{Model}_\text{pref}$.  
2. Comment on the estimation output (as defined previously).  

3. Compare $\text{Model}_\text{pref}$ and **Model 2** (both specifications) with an appropriate statistical test. Justify your choice of test. State the null hypothesis and the result of the test. Denote the preferred model as $\text{Model}_\text{pref}$.  

---

**2.0. – Model extension: interaction with socioeconomic variable**  
- Choose a socioeconomic characteristic from the dataset (e.g., age, gender).
- Create **two different model specifications**:
  - One where the characteristic interacts with an **alternative-specific constant**.
  - One where it interacts with an **attribute** (e.g., cost or time).


In [33]:
# Define your two Model 2 specifications here.
# Choose the socioeconomic characteristic and interactions yourself.
# Estimate, interpret, and compare the models

# Ensure 'df' is the DataFrame (defensive: avoid any accidental overwrite)
if not isinstance(df, pd.DataFrame):
    df = pd.read_table('lpmc25.dat')
    df['total_pt_duration'] = df['dur_pt_access'] + df['dur_pt_rail'] + df['dur_pt_bus'] + df['dur_pt_int']
    df['total_driving_cost'] = df['cost_driving_fuel'] + df['cost_driving_ccharge']
    database = db.Database('db25', df)

In [35]:
'''
Base model: Model_pref = Model 1, add Socioeconomic characteristic: female

Two specifications are DEFINED here:
   (A) female interacted with the alternative-specific constants  (1 = female, 0 = male)
   (B) female interacted with the time attribute

Setting all new interaction parameters to 0 recovers Model 1, so the LRT is valid.
'''

# Socioeconomic variable 
female = Variable('female')

# parameters (same names and init values as in Model 1) 
b_time = Beta('b_time', -6.90, None, None, 0)
delta_time_walk  = Beta('delta_time_walk', -0.94, None, None, 0)
delta_time_cycle = Beta('delta_time_cycle', 0.20, None, None, 0)
delta_time_pt    = Beta('delta_time_pt', 3.43, None, None, 0)
b_cost = Beta('b_cost', -0.190, None, None, 0)

asc_walk  = Beta('asc_walk',  0.0,    None, None, 1)   # ref
asc_cycle = Beta('asc_cycle', -4.89,  None, None, 0)
asc_pt    = Beta('asc_pt',    -2.08,  None, None, 0)
asc_car   = Beta('asc_car',   -1.61,  None, None, 0)

# A: female interacted with ASCs
asc_cycle_female = Beta('asc_cycle_female', 0, None, None, 0)
asc_pt_female    = Beta('asc_pt_female',    0, None, None, 0)
asc_car_female   = Beta('asc_car_female',   0, None, None, 0)

V_walk_2a  = asc_walk  + (b_time + delta_time_walk) * dur_walking
V_cycle_2a = asc_cycle + asc_cycle_female * female  + (b_time + delta_time_cycle) * dur_cycling
V_pt_2a    = asc_pt + asc_pt_female * female + (b_time + delta_time_pt) * total_pt_duration + b_cost * cost_transit
V_car_2a   = asc_car + asc_car_female * female + b_time * dur_driving + b_cost * total_driving_cost
V_2a = {1: V_walk_2a, 2: V_cycle_2a, 3: V_pt_2a, 4: V_car_2a}

# B: female interacted with the time attribute
b_time_female = Beta('b_time_female', 0, None, None, 0)

V_walk_2b  = asc_walk  + (b_time + delta_time_walk  + b_time_female * female) * dur_walking
V_cycle_2b = asc_cycle + (b_time + delta_time_cycle + b_time_female * female) * dur_cycling
V_pt_2b    = asc_pt    + (b_time + delta_time_pt    + b_time_female * female) * total_pt_duration + b_cost * cost_transit
V_car_2b   = asc_car   + (b_time + b_time_female * female) * dur_driving + b_cost * total_driving_cost
V_2b = {1: V_walk_2b, 2: V_cycle_2b, 3: V_pt_2b, 4: V_car_2b}

# Availability (all alternatives available to all people)
av_2 = {1: 1, 2: 1, 3: 1, 4: 1}

# Display the two specifications(can be omitted)
print("Specification A - female x ASC:")
print("  V_walk  = asc_walk + (b_time + delta_time_walk) * dur_walking")
print("  V_cycle = asc_cycle + asc_cycle_female*female + (b_time + delta_time_cycle)*dur_cycling")
print("  V_pt    = asc_pt + asc_pt_female*female + (b_time + delta_time_pt)*total_pt_duration + b_cost*cost_transit")
print("  V_car   = asc_car + asc_car_female*female + b_time*dur_driving + b_cost*total_driving_cost")
print()
print("Specification B - female x time:")
print("  V_walk  = asc_walk + (b_time + delta_time_walk + b_time_female*female)*dur_walking")
print("  V_cycle = asc_cycle + (b_time + delta_time_cycle + b_time_female*female)*dur_cycling")
print("  V_pt    = asc_pt + (b_time + delta_time_pt + b_time_female*female)*total_pt_duration + b_cost*cost_transit")
print("  V_car   = asc_car + (b_time + b_time_female*female)*dur_driving + b_cost*total_driving_cost")

Specification A - female x ASC:
  V_walk  = asc_walk + (b_time + delta_time_walk) * dur_walking
  V_cycle = asc_cycle + asc_cycle_female*female + (b_time + delta_time_cycle)*dur_cycling
  V_pt    = asc_pt + asc_pt_female*female + (b_time + delta_time_pt)*total_pt_duration + b_cost*cost_transit
  V_car   = asc_car + asc_car_female*female + b_time*dur_driving + b_cost*total_driving_cost

Specification B - female x time:
  V_walk  = asc_walk + (b_time + delta_time_walk + b_time_female*female)*dur_walking
  V_cycle = asc_cycle + (b_time + delta_time_cycle + b_time_female*female)*dur_cycling
  V_pt    = asc_pt + (b_time + delta_time_pt + b_time_female*female)*total_pt_duration + b_cost*cost_transit
  V_car   = asc_car + (b_time + b_time_female*female)*dur_driving + b_cost*total_driving_cost


**2.1. – Behavioral interpretation**  
- Explain what each interaction is capturing in terms of behavior.
- How does this differ from $\text{Model}_\text{pref}$?
- What new insight does each interaction provide?

In [ ]:
# Explain the behavioural meaning of each interaction in your own words.
# Discuss how each specification differs from your current Model_pref.

In [ ]:
'''
Behavioral interpretation of Model 2

Model 2 introduces female into Model_1 to capture gender differences. 
Model 2 lets the intercept (A) or the slope (B) vary with gender. Both nest Model_pref(here is Model1), so LRT is valid.

Specification A (female x ASC): changes the intercept of utility.
  - asc_cycle_female, asc_pt_female, asc_car_female capture intrinsic
    gender-specific preferences for these modes, independent of time/cost.
  - A positive asc_pt_female (or negative asc_car_female) would mean
    women, all else equal, prefer PT (resp. car) more than men.

Specification B (female x time): changes the slope of the time attribute.
  - b_time_female captures the additional sensitivity of women to time.
  - A negative b_time_female means women perceive time more negatively than men.
'''

**2.2. – Estimation and result interpretation**  
- Estimate both specifications of Model 2.
- For each:
  - Report parameter estimates, t-stats or p-values.
  - Compare null and final log-likelihoods.
  - Comment on changes in signs, magnitudes, or statistical significance compared to $\text{Model}_\text{pref}$.


In [40]:
# Store your estimated Model 2 result before comparing models.
# all_results['your_model_name'] = results_your_model_name

# comparison_table, _ = compile_estimation_results(all_results)
# display(comparison_table)
# Reuse the specifications defined in 2.0

# A (female x ASC)
logprob_2a = loglogit(util=V_2a, av=av_2, i=travel_mode)
biogeme_2a = bio.BIOGEME(database, logprob_2a)
biogeme_2a.model_name = "model_2a"
results_2a = biogeme_2a.estimate()

# B (female x time) 
logprob_2b = loglogit(util=V_2b, av=av_2, i=travel_mode)
biogeme_2b = bio.BIOGEME(database, logprob_2b)
biogeme_2b.model_name = "model_2b"
results_2b = biogeme_2b.estimate()

# Report parameter estimates, t-stats, and log-likelihoods
print("=" * 60)
print("Specification A: female x ASC")
print("=" * 60)
param_2a = get_pandas_estimated_parameters(estimation_results=results_2a)
display(param_2a['Estimated parameters'])
print("Null log-likelihood: ", 5000 * np.log(1/4))
print("Final log-likelihood:", results_2a.final_loglikelihood)
print(results_2a.short_summary())


print()
print("=" * 60)
print("Specification B: female x time")
print("=" * 60)
param_2b = get_pandas_estimated_parameters(estimation_results=results_2b)
display(param_2b['Estimated parameters'])
print("Null log-likelihood: ", 5000 * np.log(1/4))
print("Final log-likelihood:", results_2b.final_loglikelihood)
print(results_2b.short_summary())


all_results['model_2a'] = results_2a
all_results['model_2b'] = results_2b


Specification A: female x ASC


,Name,Value,Robust std err.,Robust t-stat.,Robust p-value
0,b_time,-6.907172,0.447925,-15.420362,0.000000e+00
1,delta_time_walk,-0.929258,0.501524,-1.852867,6.390142e-02
2,asc_cycle,-5.812794,0.769042,-7.558483,4.085621e-14
3,asc_cycle_female,0.958962,0.733728,1.306972,1.912221e-01
4,delta_time_cycle,0.224230,0.674344,0.332516,7.394998e-01
5,asc_pt,-2.554120,0.263298,-9.700486,0.000000e+00
6,asc_pt_female,0.493021,0.226401,2.177650,2.943208e-02
7,delta_time_pt,3.435923,0.253669,13.544931,0.000000e+00
8,b_cost,-0.189311,0.016733,-11.313644,0.000000e+00
9,asc_car,-1.993499,0.243330,-8.192573,2.220446e-16


Null log-likelihood:  -6931.471805599453
Final log-likelihood: -4104.760138748497
Results for model model_2a
Nbr of parameters:		11
Sample size:			5000
Excluded data:			0
Final log likelihood:		-4104.76
Akaike Information Criterion:	8231.52
Bayesian Information Criterion:	8303.209


Specification B: female x time


,Name,Value,Robust std err.,Robust t-stat.,Robust p-value
0,b_time,-6.764443,0.696023,-9.718702,0.000000
1,delta_time_walk,-0.935268,0.501530,-1.864829,0.062205
2,b_time_female,-0.144422,0.547939,-0.263573,0.792109
3,asc_cycle,-4.886688,0.239096,-20.438160,0.000000
4,delta_time_cycle,0.197014,0.672502,0.292957,0.769555
5,asc_pt,-2.082859,0.138735,-15.013239,0.000000
6,delta_time_pt,3.428779,0.253639,13.518340,0.000000
7,b_cost,-0.189687,0.016740,-11.331093,0.000000
8,asc_car,-1.611057,0.135486,-11.890995,0.000000


Null log-likelihood:  -6931.471805599453
Final log-likelihood: -4107.303226763139
Results for model model_2b
Nbr of parameters:		9
Sample size:			5000
Excluded data:			0
Final log likelihood:		-4107.303
Akaike Information Criterion:	8232.606
Bayesian Information Criterion:	8291.261



**2.3. – Model comparisons and updated Modelpref**  
- Use appropriate statistical tests to compare $\text{Model}_\text{pref}$ with each specification of Model 2.
- For each test:
  - Justify your choice of test,
  - State the null hypothesis,
  - Report the result of the test,
  - Draw a conclusion and update the preferred model if applicable.

In [41]:
# Perform and interpret only statistically valid comparisons for your Model 2 specifications.
# State the null hypothesis and update Model_pref when justified.

# log-likelihoods and parameter counts
LL_pref = all_results['model_1'].final_loglikelihood
K_pref  = all_results['model_1'].number_of_parameters

LL_2a = all_results['model_2a'].final_loglikelihood
K_2a  = all_results['model_2a'].number_of_parameters

LL_2b = all_results['model_2b'].final_loglikelihood
K_2b  = all_results['model_2b'].number_of_parameters

N = 5000

# Model 1 vs Model A (female x ASC)
print("=" * 60)
print("TEST 1: Model 1 (pref) vs Model 2A (female x ASC)")
print("=" * 60)

# Justification
print("Test used: Likelihood Ratio Test (LRT)")
print("Justification: Model 2A is nested in Model 1 (setting the")
print("   three female-ASC interaction parameters to 0 recovers")
print("   Model 1). LRT is the appropriate test for nested models.")

# Null hypothesis
print()
print("Null hypothesis H0:")
print("   asc_cycle_female = asc_pt_female = asc_car_female = 0")
print("   (i.e. gender does not affect the intrinsic preference for any mode)")

# LRT
LR_A   = -2.0 * (LL_pref - LL_2a)
df_A   = K_2a - K_pref
crit_A = chi2.ppf(0.95, df_A)
p_A    = 1.0 - chi2.cdf(LR_A, df_A)

# result
print()
print("Result of the test:")
print(f"   LL(Model 1)  = {LL_pref:.4f}  ({K_pref} parameters)")
print(f"   LL(Model 2A) = {LL_2a:.4f}  ({K_2a} parameters)")
print(f"   LR statistic = {LR_A:.4f}")
print(f"   Degrees of freedom = {df_A}")
print(f"   Chi-squared 95% critical value = {crit_A:.4f}")
print(f"   p-value = {p_A:.6e}")

print()
if LR_A > crit_A:
    print("Conclusion: REJECT H0. Model 2A significantly improves the fit")
    print("   over Model 1 at the 5% level. Model 2A is preferred.")
    sig_A = True
else:
    print("Conclusion: FAIL TO REJECT H0. Model 2A does not significantly")
    print("   improve the fit. Model 1 remains preferred over 2A.")
    sig_A = False

# Model 1 vs Model B (female x time)
print()
print("=" * 60)
print("TEST 2: Model 1 (pref) vs Model 2B (female x time)")
print("=" * 60)

# Justification
print("Test used: Likelihood Ratio Test (LRT)")
print("Justification: Model 2B is nested in Model 1 (setting the")
print("   single female-time interaction parameter to 0 recovers")
print("   Model 1). LRT is the appropriate test for nested models.")

# Null hypothesis 
print()
print("Null hypothesis H0:")
print("   b_time_female = 0")
print("   (i.e. gender does not affect the sensitivity to travel time)")

# LRT
LR_B   = -2.0 * (LL_pref - LL_2b)
df_B   = K_2b - K_pref
crit_B = chi2.ppf(0.95, df_B)
p_B    = 1.0 - chi2.cdf(LR_B, df_B)

# result 
print()
print("Result of the test:")
print(f"   LL(Model 1)  = {LL_pref:.4f}  ({K_pref} parameters)")
print(f"   LL(Model 2B) = {LL_2b:.4f}  ({K_2b} parameters)")
print(f"   LR statistic = {LR_B:.4f}")
print(f"   Degrees of freedom = {df_B}")
print(f"   Chi-squared 95% critical value = {crit_B:.4f}")
print(f"   p-value = {p_B:.6e}")

print()
if LR_B > crit_B:
    print("Conclusion: REJECT H0. Model 2B significantly improves the fit")
    print("   over Model 1 at the 5% level. Model 2B is preferred.")
    sig_B = True
else:
    print("Conclusion: FAIL TO REJECT H0. Model 2B does not significantly")
    print("   improve the fit. Model 1 remains preferred over 2B.")
    sig_B = False

TEST 1: Model 1 (pref) vs Model 2A (female x ASC)
Test used: Likelihood Ratio Test (LRT)
Justification: Model 2A is nested in Model 1 (setting the
   three female-ASC interaction parameters to 0 recovers
   Model 1). LRT is the appropriate test for nested models.

Null hypothesis H0:
   asc_cycle_female = asc_pt_female = asc_car_female = 0
   (i.e. gender does not affect the intrinsic preference for any mode)

Result of the test:
   LL(Model 1)  = -4107.3427  (8 parameters)
   LL(Model 2A) = -4104.7601  (11 parameters)
   LR statistic = 5.1652
   Degrees of freedom = 3
   Chi-squared 95% critical value = 7.8147
   p-value = 1.600951e-01

Conclusion: FAIL TO REJECT H0. Model 2A does not significantly
   improve the fit. Model 1 remains preferred over 2A.

TEST 2: Model 1 (pref) vs Model 2B (female x time)
Test used: Likelihood Ratio Test (LRT)
Justification: Model 2B is nested in Model 1 (setting the
   single female-time interaction parameter to 0 recovers
   Model 1). LRT is the appro

In [42]:
print("Update of Model_pref")

def bic(LL, K, N):
    return -2.0 * LL + K * np.log(N)

if sig_A and sig_B:
    # Both significant. But 2A and 2B are NOT nested in each other,
    # so LRT is invalid here. Use BIC (works for non-nested models).
    b2a = bic(LL_2a, K_2a, N)
    b2b = bic(LL_2b, K_2b, N)
    print("Both specifications are significant.")
    print("Since 2A and 2B are not nested in each other, the LRT cannot")
    print("be used to compare them. We use BIC instead (lower is better).")
    print(f"   BIC(Model 2A) = {b2a:.4f}")
    print(f"   BIC(Model 2B) = {b2b:.4f}")
    if b2a < b2b:
        print("Conclusion: BIC is lower for Model 2A -> Model_pref = Model 2A")
        Model_pref_name = 'model_2a'
    else:
        print("Conclusion: BIC is lower for Model 2B -> Model_pref = Model 2B")
        Model_pref_name = 'model_2b'
elif sig_A:
    print("Only Model 2A is significant -> Model_pref = Model 2A")
    Model_pref_name = 'model_2a'
elif sig_B:
    print("Only Model 2B is significant -> Model_pref = Model 2B")
    Model_pref_name = 'model_2b'
else:
    print("Neither 2A nor 2B is significant -> Model_pref remains Model 1")
    Model_pref_name = 'model_1'

print()
print(f">>> Final Model_pref = {Model_pref_name} <<<")

Update of Model_pref
Neither 2A nor 2B is significant -> Model_pref remains Model 1

>>> Final Model_pref = model_1 <<<


# Model 3 

---

Using $\text{Model}_\text{pref}$ as the base model, include an appropriate non-linear transformation on the time variable. Report both the specification and the estimation results (as defined previously).

1. State the behavioral assumption of the non-linear specification defined in this situation and how this specification explains behavior differently from $\text{Model}_\text{pref}$.  

2. Comment on the estimation output (as defined previously).  

3. Compare $\text{Model}_\text{pref}$ and **Model 3** with an appropriate statistical test. Justify your choice of test. State the null hypothesis and the result of the test. Denote the preferred model as $\text{Model}_\text{pref}$.  

---

**3.0. – Model extension: non-linear transformation**  
- Apply a non-linear transformation to the time variable in a behaviorally justified way.
  - Examples include:
    - Logarithmic: $\log(\text{time})$
    - Piecewise linear functions
    - Squared or thresholded terms

**3.1. – Behavioral interpretation**  
- Explain the motivation behind using a non-linear transformation.
- What behavioral mechanism does this better represent?

**3.2. – Estimation and output analysis**  
- Estimate the model using the transformed specification.
- Report:
  - Parameter estimates,
  - Statistical significance,
  - Null and final log-likelihoods.
- Compare qualitatively with $\text{Model}_\text{pref}$.

In [ ]:
# Store your estimated Model 3 result before comparing models.
# all_results['your_model_name'] = results_your_model_name

# comparison_table, _ = compile_estimation_results(all_results)
# display(comparison_table)

**3.3. – Model comparison and updated Modelpref**  
- Use appropriate statistical tests to compare $\text{Model}_\text{pref}$ with each specification of Model 3.
- For each test:
  - Justify your choice of test,
  - State the null hypothesis,
  - Report the result of the test,
  - Draw a conclusion and update the preferred model if applicable.

# Model 4 

---

Using $\text{Model}_\text{pref}$ as the base model, propose and test both a **nested** and a **cross-nested** structure. Report both structures using graphs, together with the specification and the estimation results (parameter values, t-tests or p-values, null and final log likelihoods).

1. State the behavioral assumptions of the proposed nesting structures in this specific situation.  

2. Comment on the estimation output (as defined previously).  

3. Compare $\text{Model}_\text{pref}$, the nested model, and the cross-nested model using appropriate statistical tests. Justify your choice of test(s). State the null hypotheses and the result of the tests. Denote the preferred model as $\text{Model}_\text{pref}$.  

---

**4.0.a – Model extension: nested logit structure**  
- Define a **nested logit model** by grouping alternatives that share unobserved factors (e.g., private vs. public modes).
- Provide a **graphical representation** of the nesting structure.
- Specify the utility functions and nest scale parameters ($\mu$) accordingly.

**4.0.b – Model extension: cross-nested logit structure**  
- Define a **cross-nested logit model**, allowing alternatives to belong to multiple nests (e.g., car might belong partially to both private and fast modes).
- Represent the cross-nesting structure with a **graph**.
- Specify the allocation parameters and scale parameters required by the model.


**4.1. – Behavioral interpretation**  
- Explain the behavioral rationale for the proposed nestings.
  - Why are certain alternatives likely to share unobserved factors?
- How does this structure improve over $\text{Model}_\text{pref}$?

**4.2. – Estimation and result interpretation**  
- Estimate both models:
  - Report parameter values, nest parameters (scale or allocation), and statistical significance.
  - Include null and final log-likelihoods.
- Discuss the estimated degree of correlation within and across nests.

In [ ]:
# Store each estimated Model 4 result before comparing models.
# all_results['your_model_name'] = results_your_model_name

# comparison_table, _ = compile_estimation_results(all_results)
# display(comparison_table)

**4.3. – Model comparisons and updated Modelpref**  
- Use appropriate statistical tests to:
  - Compare $\text{Model}_\text{pref}$ with the nested model,
  - Compare the nested model with the cross-nested model,
- For each comparison:
  - State the null hypothesis,
  - Report the test statistic and degrees of freedom,
  - Justify which model to retain as the updated $\text{Model}_\text{pref}$.

# Model 5 

---
Using the best model from Model 3 as a starting point, propose a **mixture model** that captures correlations across alternatives similar to the one modeled in the best nested structure of Model 4. Relax the independence of the error terms across alternatives. Estimate the model and report its structure and results.

1. State the behavioral assumption behind your proposed mixture specification. What type of correlation does it capture ?  

2. Comment on the estimation output (as defined previously).  

3. Compare the estimates of this model with the estimates from the best performing model from Model 4.



**5.0. – Model extension: relaxation of independence through mixtures**  
- Start from the best model obtained in Model 3.
- Propose a **mixture model** to relax the independence of error terms across alternatives.
- Your goal is to reproduce the best nested substitution patterns from Model 4, with a mixture model.
- Clearly specify the mixture structure and the utility function for each case.

**5.1. – Behavioral interpretation**  
- Explain the behavioral assumption underlying the use of a mixture.
  - What types of substitution patterns or unobserved heterogeneity does this model capture?
  - How does this differ from the assumptions in Model 4?

**5.2. – Estimation and output analysis**  
- Estimate the model using the transformed specification.
- Report:
  - Parameter estimates,
  - Statistical significance,
  - Null and final log-likelihoods.

In [ ]:
# Store your estimated Model 5 result before comparing models.
# all_results['your_model_name'] = results_your_model_name

# comparison_table, _ = compile_estimation_results(all_results)
# display(comparison_table)

**5.3. - Compare the estimates with those from the Model 4**
- Compare with an appropriate method the estimates of this model with the estimates from Model 4.

<div class="alert alert-info">

 Denote this mixture model as **$\text{Model}_\text{pref}$**, and use it as the reference model for all remaining questions (Market Shares and Forecasting).  

</div>

# Market Shares 

---

Assume that stratified random sampling was used to produce your sample. We consider the following strata:

- S1: females aged 42 years or younger  
- S2: females aged 43 years or older  
- S3: males aged 42 years or younger  
- S4: males aged 43 years or older

The table below gives the size of each category in the full population:

|               | Age < 43     | Age ≥ 43    |
|---------------|--------------|-------------|
| **Male**      | 2,926,408    | 1,379,198   |
| **Female**    | 2,841,376    | 1,519,948   |

(Source: ONS, 2015)

---

In [ ]:
# 6.1: Define the four strata, count the sample size in each one, and calculate a weight from the population and sample shares.
# Attach the appropriate stratum weight to every observation.
# 6.2: Simulate individual probabilities, multiply by the weights, then aggregate and normalize by the total weight.
# 6.3: For observed shares, group observations by observed choice, sum the weights, and normalize by the total weight.

**6.1. – Compute stratum weights**  
- Based on your sample, compute the number of individuals in each stratum (S1–S4).
- Calculate the **weight** of each stratum.
$$
\omega_g = \frac{N_g}{N} \frac{S}{S_g}=\frac{\text{share of persons in
    segment $g$ in population}}{\text{share of persons in segment $g$ in sample}}
$$
- Report the size and weight of each stratum.  



**6.2. – Predict weighted market shares using Modelpref**  

- Use $\text{Model}_\text{pref}$ and the stratum weights to compute the **predicted market share** of each transport mode.
- Also compute **confidence intervals** for the predicted shares.
- Comment: Do the results match your expectations?  

<div class="alert alert-warning">

  <strong>Warning:</strong>\
   Depending on your machine, it might take too long to use bootstrapping for confidence intervals.
  If it's the case, you can use ```use_bootstrap=False```, in ```get_betas_for_sensitivity_analysis``` to disable bootstrapping.\
  The disadvantage is that it relies on the assumption that the estimates follow a normal distribution.
</div>

**6.3. – Compare with weighted observed shares**  

- Compute the actual weighted market shares in your sample (using the actual observed choices and stratum weights).
- Compare these to the model-predicted shares.  

# Forecasting 

---

Consider the following scenarios:

1. An additional charge of £1.5 for car users  
2. A 20% decrease in public transport cost

---

In [ ]:
# 7.1: Start from a baseline simulation; change only the attribute affected by each scenario, then simulate and aggregate weighted shares again.
# 7.2: Compare the weighted car shares from the baseline and scenarios.
# 7.3: For each scenario, combine the PT probability, scenario fare, and observation weight before aggregating revenue.
# 7.4: Obtain marginal utilities of time and cost, form their ratio with consistent units, then calculate weighted averages.
# 7.5: Calculate individual probabilities and derivatives, form direct/cross elasticities, then aggregate them with the appropriate weights.

**7.1. – Simulate market shares under policy scenarios**  

- Use $\text{Model}_\text{pref}$ to predict the market shares for each of the two policy scenarios:
  - Scenario 1: increase the cost of car by £1.5
  - Scenario 2: decrease the cost of public transport by 20%
- Compare the predicted market shares to the original (baseline) shares.
- Comment: Do the results match your expectations?  


**7.2. – Policy evaluation: car reduction**  

- Which scenario is more effective at **reducing the market share of car**?
- Justify your answer based on the model results.  

**7.3. – Revenue analysis**  

- Compute the **total revenue** from public transport under each scenario.
- Identify which scenario yields the highest revenue.
- Compare with the revenue in the original situation.
- Provide an explanation for the result.  

**7.4. – Value of time**  

- Compute the **average value of time** (VOT) for car and for public transport.
- Report the results and comment on them.  

**7.5. – Elasticity analysis**  

- Compute the following **aggregate elasticities**:
  - **Direct elasticities**:
    - Car cost $\rightarrow$ car choice
    - PT cost $\rightarrow$ PT choice
  - **Cross elasticities**:
    - Car cost $\rightarrow$ PT, Cycle and Walk
    - PT cost $\rightarrow$ Car, Cycle and Walk
- A total of 8 values must be reported.
- Comment on the signs and magnitudes of the elasticities.  